In [19]:
from deap import creator, base, tools
import random
import numpy as np

from impl.mr.mr import EnvPerturbation

creator.create("Fitness", base.Fitness, weights=(1.0, -1.0))
creator.create("Individual", list, fitness=creator.Fitness)

toolbox = base.Toolbox()
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_bool, 100)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def evalOneMax(individual):
    return sum(individual),min(individual)

toolbox.register("evaluate", evalOneMax)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean, axis=0)
stats.register("std", np.std, axis=0)
stats.register("min", np.min, axis=0)
stats.register("max", np.max, axis=0)
logbook = tools.Logbook()
logbook.header = "gen", "type", "evals", "std", "min", "avg", "max"

pop = toolbox.population(n=300)
fitnesses = toolbox.map(toolbox.evaluate, pop)
for ind, fit in zip(pop, fitnesses):
    ind.fitness.values = fit

CXPB, MUTPB = 0.5, 0.2
fits = [ind.fitness.values[0] for ind in pop]
g = 0
# Begin the evolution
while max(fits) < 100 and g < 100:
    # A new generation
    g = g + 1
    # print("-- Generation %i --" % g)
    # Select the next generation individuals
    offspring = toolbox.select(pop, len(pop))
    # Clone the selected individuals
    offspring = list(toolbox.map(toolbox.clone, offspring))
    # Apply crossover and mutation on the offspring
    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < CXPB:
            toolbox.mate(child1, child2)
            del child1.fitness.values
            del child2.fitness.values

    for mutant in offspring:
        if random.random() < MUTPB:
            toolbox.mutate(mutant)
            del mutant.fitness.values

    # Evaluate the individuals with an invalid fitness
    invalid_ind = [ind for ind in offspring if not ind.fitness.valid]
    fitnesses = toolbox.map(toolbox.evaluate, invalid_ind)
    for ind, fit in zip(invalid_ind, fitnesses):
        ind.fitness.values = fit

    pop[:] = offspring

    record = stats.compile(pop)
    logbook.record(gen=g, type="a", evals=len(pop), **record)
    logbook.record(gen=g, type="b", evals=len(pop), **record)
    print(logbook.stream)

    # Gather all the fitnesses in one list and print the stats
    fits = [ind.fitness.values[0] for ind in pop]

gen	type	evals	std                    	min      	avg                      	max      
1  	a   	300  	[4.04274934 0.        ]	[43.  0.]	[54.05333333  0.        ]	[66.  0.]
1  	b   	300  	[4.04274934 0.        ]	[43.  0.]	[54.05333333  0.        ]	[66.  0.]
2  	a   	300  	[3.22731398 0.        ]	[48.  0.]	[57.13333333  0.        ]	[69.  0.]
2  	b   	300  	[3.22731398 0.        ]	[48.  0.]	[57.13333333  0.        ]	[69.  0.]
3  	a   	300  	[3.20347034 0.        ]	[50.  0.]	[59.33333333  0.        ]	[70.  0.]
3  	b   	300  	[3.20347034 0.        ]	[50.  0.]	[59.33333333  0.        ]	[70.  0.]
4  	a   	300  	[3.19391609 0.        ]	[50.  0.]	[61.83  0.  ]            	[73.  0.]
4  	b   	300  	[3.19391609 0.        ]	[50.  0.]	[61.83  0.  ]            	[73.  0.]
5  	a   	300  	[2.81744368 0.        ]	[55.  0.]	[63.80333333  0.        ]	[73.  0.]
5  	b   	300  	[2.81744368 0.        ]	[55.  0.]	[63.80333333  0.        ]	[73.  0.]
6  	a   	300  	[2.68895684 0.        ]	[57.  0.]	[65.94666667  0.

In [1]:
import carla
import random

client = carla.Client("172.30.32.1", 2000)
client.load_world("Town01")
world = client.get_world()
vehicle_blueprints = world.get_blueprint_library().filter("*vehicle*")
spawn_point = random.choice(world.get_map().get_spawn_points())
ego_vehicle = world.spawn_actor(random.choice(vehicle_blueprints), spawn_point)
pedestrian_blueprints = world.get_blueprint_library().filter("*pedestrian*")
if -10 < spawn_point.rotation.yaw < 10:
    offset = carla.Location(x=7, y=1, z=2)
elif 80 < spawn_point.rotation.yaw < 100:
    offset = carla.Location(x=-1, y=7, z=2)
elif 170 < spawn_point.rotation.yaw < 190:
    offset = carla.Location(x=-7, y=-1, z=2)
else:
    offset = carla.Location(x=1, y=-7, z=2)
pedestrian = world.spawn_actor(
    random.choice(pedestrian_blueprints),
    carla.Transform(spawn_point.location + offset, carla.Rotation(0, spawn_point.rotation.yaw - 90, 0))
)
camera_init_trans = carla.Transform(carla.Location(z=1.5))
camera_bp = world.get_blueprint_library().find("sensor.camera.rgb")
camera_bp.set_attribute("image_size_x", "832")
camera_bp.set_attribute("image_size_y", "640")
camera = world.spawn_actor(camera_bp, camera_init_trans, attach_to=ego_vehicle)
camera.listen(lambda image: image.save_to_disk('out/%06d.png' % image.frame))
camera.id
# camera = world.get_actor(199)
# camera.destroy()

375

In [ ]:
from cyber_py3 import cyber, cyber_time
import time
import os
from modules.routing.proto.routing_pb2 import RoutingRequest, LaneWaypoint


class ApolloFeatures:

    def __init__(self):
        cyber.init()
        self.node = cyber.Node("apollo_features")
        self.routing_writer = self.node.create_writer('/apollo/routing_request', RoutingRequest)

    def launch_control(self):
        os.system('cyber_launch start /apollo/modules/control/launch/control.launch &')

    def launch_routing(self):
        os.system('cyber_launch start /apollo/modules/routing/launch/routing.launch &')

    def launch_perception(self):
        os.system('cyber_launch start /apollo/modules/transform/launch/static_transform.launch &')
        os.system('cyber_launch start /apollo/modules/perception/production/launch/perception.launch &')

    def launch_prediction(self):
        os.system('cyber_launch start /apollo/modules/prediction/launch/prediction.launch &')

    def launch_rtk_localization(self):
        os.system('cyber_launch start /apollo/modules/localization/launch/rtk_localization.launch &')

    def launch_planning(self):
        os.system('cyber_launch start /apollo/modules/planning/launch/planning.launch &')

    def launch_all_modules(self):
        self.launch_rtk_localization()
        self.launch_perception()
        self.launch_prediction()
        self.launch_planning()
        self.launch_control()
        self.launch_routing()

    def send_routing_request(self, x_start, y_start, x_end, y_end):
        msg = RoutingRequest()
        msg.header.module_name = 'dreamview'
        msg.header.sequence_num = 0
        waypoint = msg.waypoint.add()
        waypoint.pose.x = float(x_start)
        waypoint.pose.y = float(y_start)
        waypoint = msg.waypoint.add()
        waypoint.pose.x = float(x_end)
        waypoint.pose.y = float(y_end)

        time.sleep(2.0)
        self.routing_writer.write(msg)
